# Lesson 3 — 도메인 피처와 상호작용

**예상 시간:** 개념 40분 + 실습 75분  
**오늘의 새 산출물:** feature group ablation과 validation 근거가 있는 최종 후보 feature set

Lesson 2까지 선택한 feature set을 기준으로 사용합니다. 실제 과제의 완성 코드는 포함하지 않습니다.

## 1. 오늘의 질문

> 아침 8시라는 정보는 평일과 휴일에 같은 의미일까? 같은 기온이라도 습도와 체감온도가 다르면 자전거 수요에 같은 조건일까?

상호작용(Interaction)은 두 조건이 함께 있을 때 한 조건의 의미가 달라지는 관계를 표현합니다.

## 2. 선수 지식 확인

- Lesson 2의 선택된 feature set과 pipeline을 재현할 수 있는가?
- feature를 추가할 때 다른 모델·fold·metric을 바꾸면 공정한 비교가 깨진다는 점을 설명할 수 있는가?
- validation 개선과 경제적 인과관계를 구분할 수 있는가?

## 3. 개념 설명

### 3.1 왜 도메인 피처가 필요한가?

원본 `hour=8`, `workingday=1` 두 열만으로도 tree 모델이 조합을 찾을 수 있지만, `workingday_morning_peak=1`을 만들면 우리가 검증하려는 가설을 명시할 수 있습니다. 이는 모델에 새로운 미래 정보를 주는 것이 아니라 이미 아는 두 정보를 다른 관점으로 표현하는 일입니다.

### 3.2 사람이 손으로 하는 작업

각 행에서 시간을 보고 시간대 이름을 붙이고, 평일이면서 아침 출근 시간인지 확인해 0/1을 적습니다. `temp_atemp_gap`은 `atemp - temp`를 계산합니다. `high_humidity`는 수업에서 고정한 기준 이상이면 1로 표시합니다.

### 3.3 Feature group과 ablation

피처 하나하나를 무질서하게 넣으면 무엇이 효과를 냈는지 알기 어렵습니다. 관련 피처를 그룹으로 묶고 기준 모델에 한 그룹씩 추가합니다. Ablation은 동일한 조건에서 특정 그룹의 포함 여부만 바꾸는 통제 비교입니다.

이번 순서는 시간대 → 근무일 상호작용 → 날씨·체감입니다. 앞 단계가 validation에서 채택되면 다음 단계의 기준으로 사용하고, 개선되지 않으면 제외한 기준에서 다음 그룹을 시험합니다.

### 3.4 유용성과 위험

상호작용은 데이터가 충분하고 행동 패턴이 반복될 때 유용할 수 있습니다. 너무 많은 임계값과 조합은 우연한 validation 패턴에 맞출 위험이 있습니다. 임계값을 점수에 맞춰 반복 조정하지 않고 수업에서 정한 값으로 한 번 비교합니다.

이번 과정에서는 `count` lag나 rolling을 만들지 않습니다. test의 미래 시간에는 직전 test target이 제공되지 않으므로 계산 가능성과 예측 운영 방식을 먼저 정의해야 하기 때문입니다.

## 4. 손으로 만드는 작은 표

| hour | workingday | humidity | weather | hour_group | morning_peak | work_morning | high_humidity |
|---:|---:|---:|---:|---|---:|---:|---:|
| 8 | 1 | 55 | 1 | morning_peak | 1 | 1 | 0 |
| 8 | 0 | 55 | 1 | morning_peak | 1 | 0 | 0 |
| 18 | 1 | 85 | 3 | evening_peak | 0 | 0 | 1 |

같은 8시라도 `workingday_morning_peak`는 근무일에만 1입니다. 이 열은 출근 가설을 직접 표현하지만 실제 도움이 되는지는 OOF RMSLE로 확인해야 합니다.

## 5. 실행 가능한 toy example

실제 과제와 다른 배달 주문 데이터에서 `np.select`와 논리 조건을 사용합니다.

In [ ]:
import numpy as np
import pandas as pd

toy = pd.DataFrame({
    'hour': [2, 8, 14, 18, 22],
    'workingday': [1, 1, 1, 1, 0],
    'temperature': [6.0, 9.0, 18.0, 13.0, 8.0],
    'feels_like': [4.0, 8.0, 18.0, 10.0, 6.0],
    'humidity': [80, 55, 45, 85, 72],
})

def add_delivery_features(frame):
    result = frame.copy()
    conditions = [
        result['hour'].between(0, 5),
        result['hour'].between(6, 9),
        result['hour'].between(10, 16),
        result['hour'].between(17, 19),
    ]
    labels = ['night', 'morning_peak', 'daytime', 'evening_peak']
    result['hour_group'] = np.select(conditions, labels, default='evening')
    result['is_morning_peak'] = result['hour'].between(7, 9).astype(int)
    result['workingday_morning_peak'] = (
        (result['workingday'] == 1) & (result['is_morning_peak'] == 1)
    ).astype(int)
    result['feels_like_gap'] = result['feels_like'] - result['temperature']
    result['high_humidity'] = (result['humidity'] >= 80).astype(int)
    return result

toy_features = add_delivery_features(toy)
print(toy_features)
print('입력 원본은 변경되지 않음:', toy.columns.tolist())
print('반환 type:', type(toy_features))

`between(left, right)`는 양 끝을 포함하는 boolean Series를 반환합니다. `.astype(int)`는 True/False를 1/0으로 바꾼 새 Series를 반환합니다. `np.select`는 여러 조건 중 처음 참인 label을 고른 새 배열을 반환합니다. 함수는 `frame.copy()`를 사용하므로 입력 원본을 바꾸지 않습니다.

## 6. Data Leakage 점검

- 새 feature가 `datetime`과 test에 제공된 원본 입력만 사용하는가?
- `count`, `casual`, `registered` 또는 그 집계값을 사용하지 않았는가?
- feature 함수가 train/validation/test에 동일하게 적용되는가?
- validation 점수를 여러 번 보며 임계값을 임의로 미세 조정하지 않았는가?
- feature importance를 원인 증거로 해석하지 않았는가?

## 7. 실제 데이터 Exercise

Lesson 2까지의 최선 set을 Base로 고정하고 아래 그룹을 순서대로 시험합니다.

1. **시간대 그룹:** `hour_group`, `is_morning_peak`(7~9시), `is_evening_peak`(17~19시)
2. **근무일 상호작용:** `workingday_morning_peak`, `workingday_evening_peak`
3. **날씨·체감 그룹:** `temp_atemp_gap = atemp - temp`, `high_humidity = humidity >= 80`, `weather_discomfort = weather >= 3`

`hour_group` 경계는 night 0~5, morning_peak 6~9, daytime 10~16, evening_peak 17~19, evening 20~23으로 고정합니다. 그룹별 결과를 본 뒤 임계값을 재조정하지 않습니다.

## 8. 코드 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/code/lesson3.ipynb`

- **C1 — Feature 계약:** Base와 세 feature group의 이름, 계산식, source 컬럼, prediction time 가용성을 표로 만든다.
- **C2 — 순수 feature 함수:** 입력을 변경하지 않고 지정 그룹 피처를 추가한 새 DataFrame을 반환한다. 원본/결과 컬럼과 결측치 수를 출력한다.
- **C3 — 단계별 ablation:** Base에서 시작해 시간대 그룹을 비교한다. OOF RMSLE가 낮아진 경우에만 채택한다. 그 결과를 기준으로 근무일, 날씨 그룹을 같은 방식으로 순차 비교한다.
- **C4 — 비교표:** 단계, 포함 그룹, feature 수, fold별 RMSLE/MAE, OOF RMSLE/MAE, 채택 여부를 한 표로 출력한다.
- **C5 — 최종 후보:** 채택된 feature 이름 목록과 제외된 그룹 목록을 출력하고 같은 입력 행 수가 유지됐는지 확인한다.

**코드 최소 통과 기준**

- 수업에서 고정한 시간·습도·날씨 임계값을 사용한다.
- 한 단계에서 feature group 외의 모델·fold·지표가 바뀌지 않는다.
- 모든 pipeline은 fold train에서 새로 fit된다.
- target-derived feature가 없다.
- 각 채택 여부가 OOF RMSLE로 결정된다.

## 9. 글 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/text/lesson3.txt`

- **T1:** `2012-10-16 08:00:00`을 예로 `hour`, `workingday`, `workingday_morning_peak`의 차이를 설명한다. **통과 기준:** 원본 단서와 상호작용 표현을 구분한다.
- **T2:** 각 그룹의 OOF RMSLE를 수치로 적고 채택·제외를 설명한다. **통과 기준:** validation 개선이 없는 그룹도 정직하게 제외한다.
- **T3:** target lag를 이번 과정에서 제외한 이유를 설명한다. **통과 기준:** test 미래 시점의 이전 target 가용성과 운영 방식이 정의되지 않았음을 언급한다.
- **T4:** 상호작용 feature가 성능을 개선했더라도 출퇴근이 수요 증가의 원인이라고 단정할 수 없는 이유를 설명한다. **통과 기준:** 예측 관계와 인과관계를 구분한다.

## 10. 성찰 질문

**제출 불필요:** feature를 열 개 더 만들었는데 점수가 그대로라면 실패일까요, 아니면 불필요한 복잡성을 피하게 해 준 유용한 실험일까요?

## 11. 제출 전 자체 점검

- [ ] C1~C5, T1~T4가 모두 있는가?
- [ ] feature 함수가 원본을 변경하지 않는가?
- [ ] 그룹을 한 번에 하나씩 비교했는가?
- [ ] validation 점수에 맞춰 임계값을 반복 조정하지 않았는가?
- [ ] 예측력과 인과관계를 구분했는가?